<a href="https://colab.research.google.com/github/mdudavillela/gradient-convergence-mlp/blob/main/notebooks/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main Notebook

Este notebook coordena o projeto: ele não tem a lógica pesada em si, só chama funções que moram nos módulos dentro de `src/` (mesmo papel que um `main.R` com `source()` cumpriria se fosse programado em R).

## 1. Clonar o repositório

O colab roda numa máquina virtual temporária, ela não tem acesso ao repositório do github por padrão, então o primeiro passo é baixar uma cópia dele pra dentro da máquina (`src/coleta.py`).

- `!git clone {REPO_URL}` — o `!` diz ao notebook "isso não é python, é um comando de terminal". É o mesmo `git clone` que seria usado fora do colab.
- `%cd {REPO_DIR}` — o `%` marca um comando "mágico" do Jupyter. Esse em específico muda a pasta de trabalho pra dentro do repositório clonado, pra que os caminhos relativos usados nas próximas células (tipo `"src"` ou `"data/tmdb_raw.csv"`) funcionem certinho.

In [1]:
import os  # biblioteca pra interagir com o sistema operacional

REPO_URL = "https://github.com/mdudavillela/gradient-convergence-mlp.git"
REPO_DIR = "gradient-convergence-mlp"  # nome da pasta que o clone cria (sempre o nome do repositório)
CAMINHO_REPO = f"/content/{REPO_DIR}"  # caminho absoluto, não depende de onde o notebook "está" no momento

# só clona se a pasta ainda não existir, evita erro caso essa célula rode duas vezes na mesma sessão
if not os.path.exists(CAMINHO_REPO):
    %cd /content
    !git clone {REPO_URL}
else:
    print(f"Pasta '{CAMINHO_REPO}' já existe - pulando o clone.")

%cd {CAMINHO_REPO}

/content
Cloning into 'gradient-convergence-mlp'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 71 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 222.41 KiB | 1.71 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/gradient-convergence-mlp


## 2. Importar o módulo de coleta

`sys.path` é a lista de pastas onde o python procura por módulos quando se escreve `import algo`. Por padrão, `src/` não está nessa lista. Sem esse passo, `from coleta import coletar_dataset` daria `ModuleNotFoundError`.

In [2]:
import sys
sys.path.append("src")

from coleta import coletar_dataset

print("Módulo 'coleta' importado com sucesso.")

Módulo 'coleta' importado com sucesso.


## 3. Configurar a API key

Tenta carregar a chave dos **Secrets** do colab primeiro (configurar uma vez: ícone 🔑 na barra lateral → "Add new secret" → nome `TMDB_API_KEY`).

In [3]:
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("TMDB_API_KEY")
    print("API key carregada dos Secrets do colab.")
except Exception:
    api_key = getpass.getpass("Cole sua API Key (v3 auth) do TMDB: ")

API key carregada dos Secrets do colab.


## 4. Rodar a coleta

Aqui chamamos a função "chefe" do `coleta.py`. O notebook não precisa saber como a coleta funciona por dentro, só precisa saber o nome da função, quais parâmetros ela espera, e o que ela devolve (um `DataFrame`). Essa separação entre "o que fazer" (aqui) e "como fazer" (dentro do módulo) é o ganho principal de organizar o código assim.

Na primeira vez, demora alguns minutos (milhares de requisições à API). Depois disso, mesmo numa sessão nova do colab, se `data/tmdb_raw.csv` já estiver commitado no repositório, essa célula carrega ele na hora em vez de recoletar (é a lógica de cache de dentro da `coletar_dataset`).


In [4]:
df_raw = coletar_dataset(
    api_key = api_key,
    cache_path = "data/tmdb_raw.csv",
    n_paginas = 250,          # ~20 filmes por página -> até ~5000 filmes candidatos
    qtd_votos_min = 100,      # mínimo de votos para a nota do público ser confiável
    numero_threads = 10,      # requisições simultâneas na busca de detalhes
    atualizar_do_zero = False, # mudar para True só se quiser recoletar dados atualizados -> NUNCA DURANTE A IC
)

print(df_raw.shape)
df_raw.head()

Cache encontrado em 'data/tmdb_raw.csv' — carregando em vez de recoletar.
(4690, 12)


,id,title,budget,revenue,runtime,popularity,vote_average,vote_count,release_date,original_language,genres,production_companies_count
0,1204680,Coyote vs. Acme,72000000,88043208,103,290.8185,7.525,632,2026-08-17,en,"['Comedy', 'Adventure', 'Family']",4
1,1101383,The End of Oak Street,80000000,120363694,100,235.5727,7.036,1785,2026-08-12,en,"['Science Fiction', 'Mystery', 'Thriller']",4
2,1423191,Resident Evil,75000000,212394895,95,433.5555,7.308,694,2026-09-16,en,"['Horror', 'Science Fiction', 'Adventure']",7
3,969681,Spider-Man: Brand New Day,225000000,2495683000,145,564.5909,7.900,3000,2026-07-29,en,"['Science Fiction', 'Action', 'Adventure']",4
4,1032863,The Love Hypothesis,0,0,112,205.4903,8.039,441,2026-09-23,en,"['Romance', 'Comedy']",3


## 5. Próximo passo: levar o CSV para o repositório

Se a coleta acabou de rodar pela primeira vez, `data/tmdb_raw.csv` existe só nesta máquina temporária do colab e assim que a sessão desconectar, ele some, porque ainda não está no github.

Jeito mais simples: baixar o arquivo do colab (ícone de pasta 📁 na lateral → botão direito em `data/tmdb_raw.csv` → Download) e adicionar ele ao repositório do mesmo jeito que foi feito com o `coleta.py` (Add file → Create new file ou Upload files → commit).

Depois disso, qualquer sessão futura (minha ou de outra pessoa) já encontra o cache pronto sem precisar de API key nem esperar os minutos de coleta de novo.

## 6. Importar o módulo de pré-processamento


In [5]:
import sys
import os

from preprocessamento import (
    carregar_dados_validos,
    criar_alvo,
    selecionar_features,
    tratar_categoricas,
    dividir_e_normalizar,
)

print("Módulo 'preprocessamento' importado com sucesso.")

Módulo 'preprocessamento' importado com sucesso.


## 7. Rodar o pré-processamento
Encadeando as funções na ordem do pipeline:

`tmdb_raw.csv -> carregar_dados_validos() -> criar_alvo() -> selecionar_features() ->
tratar_categoricas() -> dividir_e_normalizar() -> (x_treino, x_teste, y_treino, y_teste)`

In [6]:
df = carregar_dados_validos("data/tmdb_raw.csv")
df = criar_alvo(df)

x_numerico = selecionar_features(df)
x_categorico = tratar_categoricas(df)
y = df["log_revenue"].values

x_treino, x_teste, y_treino, y_teste, nomes_colunas = dividir_e_normalizar(x_numerico, x_categorico, y)

print("Treino:", x_treino.shape)
print("Teste :", x_teste.shape)
print("Colunas:", nomes_colunas)

Treino: (2988, 20)
Teste : (748, 20)
Colunas: ['log_budget', 'runtime', 'release_year', 'production_companies_count', 'genero_Drama', 'genero_Comedy', 'genero_Action', 'genero_Thriller', 'genero_Adventure', 'genero_Crime', 'genero_Science Fiction', 'genero_Fantasy', 'genero_Romance', 'genero_Horror', 'idioma_en', 'idioma_es', 'idioma_fr', 'idioma_ja', 'idioma_ko', 'idioma_outro']
